In [1]:
from ta.volatility import AverageTrueRange

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from backtesting import Strategy
from backtesting import Backtest, set_bokeh_output
set_bokeh_output(notebook=False)

from datetime import timedelta

import sys
sys.path.append('../utils')
import Util
import importlib
importlib.reload(Util)

from enum import Enum
import time

from random import randint
import random


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [9]:
pair = 'CADCHF'

df_d1_raw = pd.read_csv(f"../Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h4_raw = pd.read_csv(f"../Trading Data/{pair}_H4.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_m15_raw = pd.read_csv(f"../Trading Data/{pair}_M15.csv",sep=',', parse_dates=['Date'], index_col='Date')

print(df_d1_raw)
print(df_h4_raw)
print(df_m15_raw)

               Open     High      Low    Close  Tickvol
Date                                                   
2010-09-23  0.95790  0.95845  0.94550  0.95260    81300
2010-09-24  0.95260  0.96180  0.94780  0.95980   103852
2010-09-27  0.96055  0.96420  0.95510  0.95655   115385
2010-09-28  0.95655  0.95765  0.94310  0.94775   126433
2010-09-29  0.94765  0.95425  0.94400  0.94620   115335
...             ...      ...      ...      ...      ...
2026-10-05  0.58075  0.58381  0.58046  0.58250    48951
2026-10-06  0.58249  0.58551  0.58186  0.58538    42990
2026-10-07  0.58540  0.58631  0.58206  0.58423    50257
2026-10-08  0.58424  0.58563  0.58294  0.58443    66525
2026-10-09  0.58444  0.58472  0.58121  0.58234    34325

[4185 rows x 5 columns]
                        Open     High      Low    Close  Tickvol
Date                                                            
2010-09-23 08:00:00  0.95790  0.95845  0.95285  0.95370    19153
2010-09-23 12:00:00  0.95370  0.95455  0.94550  0.95

In [10]:
structure_d1_window = 2
structure_m15_window = 2

date_format = '%Y-%m-%d %H:%M'
# start_date = '2023-1-1 00:00'
# end_date = '2024-3-29 00:00'
start_date = datetime.strptime('2021-1-1 00:00', date_format)
end_date = datetime.strptime('2026-9-16 00:00', date_format)
# start_date = '2022-01-1 00:00'

timezone = "Europe/London"

# nombre de jours pour les high.low
define_key_level_window = 90

gap_look_ahead = 6

# nombre de candle au dessus/ en desous de key level après breakout
breakout_candles = 9

# nombre de jours à chercher key level
key_level_lookback_candle = 100

# vérifier si 30 candle dans ke passé pour valider un valid breakout,
# un valid breakout doit avoir {breakout_candles} candles au dessus du resistant level
breakout_candles_lookback = 30

start_trade_hour = 9
stop_trade_hour = 19

atr_multiplier = 0.7
min_touched = 3

In [11]:
# process D1 data
df_d1 = df_d1_raw.loc[start_date - timedelta(days=30) : end_date]

df_d1 = Util.implement_market_structure_states_fixed_pivots(df_d1, n=structure_d1_window)
df_d1["state_changed"] = df_d1["market_state"] != df_d1[
    "market_state"
].shift(1)
for state in df_d1[df_d1["state_changed"]][["Close", "market_state"]].iterrows():
    print(state)
df_d1 = df_d1.loc[start_date : end_date]

(Timestamp('2020-12-02 00:00:00'), Close           0.69255
market_state        NaN
Name: 2020-12-02 00:00:00, dtype: object)
(Timestamp('2020-12-03 00:00:00'), Close           0.69245
market_state        NaN
Name: 2020-12-03 00:00:00, dtype: object)
(Timestamp('2020-12-04 00:00:00'), Close           0.69728
market_state        NaN
Name: 2020-12-04 00:00:00, dtype: object)
(Timestamp('2020-12-07 00:00:00'), Close           0.6955
market_state       NaN
Name: 2020-12-07 00:00:00, dtype: object)
(Timestamp('2020-12-08 00:00:00'), Close           0.69335
market_state        NaN
Name: 2020-12-08 00:00:00, dtype: object)
(Timestamp('2020-12-09 00:00:00'), Close           0.6932
market_state       NaN
Name: 2020-12-09 00:00:00, dtype: object)
(Timestamp('2020-12-10 00:00:00'), Close           0.69495
market_state        NaN
Name: 2020-12-10 00:00:00, dtype: object)
(Timestamp('2020-12-11 00:00:00'), Close           0.6964
market_state       NaN
Name: 2020-12-11 00:00:00, dtype: object)
(Times

In [34]:
# process h4 data
df_h4 = df_h4_raw.loc[start_date - timedelta(days=20) : end_date]
df_h4 = Util.implement_market_structure_states_fixed_pivots(df_h4, 5)
indicator_atr_14 = AverageTrueRange(df_h4['High'], df_h4['Low'], df_h4['Close'], window=14)
df_h4['ATR'] = indicator_atr_14._atr
df_h4 = df_h4.loc[start_date : end_date]

h4_d1_direction_list = []
for i in range(len(df_d1)):
    row = df_d1.iloc[i]
    start_date_daily = row.name
    end_date_daily = start_date_daily + timedelta(hours=23, minutes=59)
    df_by_day = df_h4.loc[f'{start_date_daily}' : f'{end_date_daily}']

    for index2, row2 in df_by_day.iterrows():
        h4_d1_direction_list.append(df_d1.iloc[i-1]['market_state'])

df_h4['market_state_d1'] = h4_d1_direction_list

df_h4

,Open,High,Low,Close,Tickvol,is_swing_high,is_swing_low,market_state,ATR,market_state_d1
Date,,,,,,,,,,
2021-01-04 00:00:00,0.69378,0.69573,0.69310,0.69555,12960,False,False,BULLISH,0.001934,BULLISH
2021-01-04 04:00:00,0.69553,0.69625,0.69513,0.69527,9826,True,False,BULLISH,0.001876,BULLISH
2021-01-04 08:00:00,0.69525,0.69580,0.69311,0.69377,20631,False,False,BULLISH,0.001934,BULLISH
2021-01-04 12:00:00,0.69377,0.69405,0.69151,0.69293,23712,False,False,BULLISH,0.001977,BULLISH
2021-01-04 16:00:00,0.69296,0.69395,0.68858,0.68984,32860,False,False,BULLISH,0.002220,BULLISH
...,...,...,...,...,...,...,...,...,...,...
2026-09-15 08:00:00,0.58789,0.58839,0.58749,0.58791,6852,False,False,BULLISH,0.001281,BULLISH
2026-09-15 12:00:00,0.58792,0.58904,0.58735,0.58878,9087,False,False,BULLISH,0.001310,BULLISH
2026-09-15 16:00:00,0.58879,0.58880,0.58812,0.58812,4546,False,False,BULLISH,0.001265,BULLISH


In [35]:
resistants_list = Util.define_resistant_level_with_atr(df_h4, define_key_level_window, atr_multiplier=atr_multiplier, min_touched=min_touched)
supports_list = Util.define_support_level_with_atr(df_h4, define_key_level_window, atr_multiplier=atr_multiplier, min_touched=min_touched)

df_h4['support_point'] = df_h4.apply(lambda row: Util.support_pos(supports_list, row), axis=1)
df_h4['is_support'] = df_h4.apply(lambda row: Util.is_support(supports_list, row), axis=1)

df_h4['resistant_point'] = df_h4.apply(lambda row: Util.resistant_pos(resistants_list, row), axis=1)
df_h4['is_resistant'] = df_h4.apply(lambda row: Util.is_resistant(resistants_list, row), axis=1)

df_h4['valid_breakout_support_levels'] = Util.is_candles_above_support_level_list(df_h4, breakout_candles, key_level_lookback_candle)
df_h4['valid_breakout_resistant_levels'] = Util.is_candles_below_resistant_level_list(df_h4, breakout_candles, key_level_lookback_candle)


In [58]:
# using m15 market structure for entry
def apply_total_signal(df):
    # Initialize the 'TotalSignal' column
    df_signal_list = []
    valid_support_levels_list = []
    valid_resistant_levels_list = []

    for i in range(0, len(df)):
        valid_breakout_resistant_levels = []
        valid_breakout_support_levels = []
        signal = 0
        # valid resistants levels are when valid breakout and levels touched
        valid_resistant_level = None
        valid_support_levels = None

        # short
        # breakout_candles_below candle needs to be below resistant level for the breakout to be valid
        valid_breakout_resistant_levels = df['valid_breakout_resistant_levels'].iloc[i-1]
        if len(valid_breakout_resistant_levels) > 0:
            valid_resistant_level = min(valid_breakout_resistant_levels)
            signal = 1

        # long
        # breakout_candles_above candle needs to be above support level for the breakout to be valid
        valid_breakout_support_levels = df['valid_breakout_support_levels'].iloc[i-1]
        if len(valid_breakout_support_levels) > 0:
            valid_support_levels = max(valid_breakout_support_levels)
            signal = 2

        df_signal_list.append(signal)
        valid_resistant_levels_list.append(valid_resistant_level)
        valid_support_levels_list.append(valid_support_levels)

    return df_signal_list, valid_resistant_levels_list, valid_support_levels_list

df_h4['TotalSignal'], df_h4['valid_resistant_levels'], df_h4['valid_support_levels'] = apply_total_signal(df=df_h4)

In [59]:
len(df_h4[df_h4.TotalSignal != 0])

3200

In [38]:
df_h4['pointpos'] = df_h4.apply(lambda row: Util.pointpos(row), axis=1)

In [18]:
start_date_chart = '2022-1-1 00:00'
end_date_chart = '2022-12-30 00:00'
dfpl = df_h4.loc[start_date_chart : end_date_chart]#.set_index("Gmt time")

# Create a plot with 2 rows
fig = make_subplots(rows=1, cols=1)

# Add candlestick plot on the first row
fig.add_trace(go.Candlestick(x=dfpl.index,
                             open=dfpl['Open'],
                             high=dfpl['High'],
                             low=dfpl['Low'],
                             close=dfpl['Close']),
              row=1, col=1)

# Add markers for entry point m15
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['pointpos'], mode="markers",
                         marker=dict(size=12, color="MediumPurple"),
                         name="entry"),
              row=1, col=1)

fig.update_layout(width=1500, height=800, sliders=[])
fig.show()

In [19]:
start_date_chart = '2022-1-1 00:00'
end_date_chart = '2022-12-30 00:00'
dfpl = df_h4.loc[start_date_chart : end_date_chart]#.set_index("Gmt time")

# Create a plot with 2 rows
fig = make_subplots(rows=1, cols=1)

# Add candlestick plot on the first row
fig.add_trace(go.Candlestick(x=dfpl.index,
                             open=dfpl['Open'],
                             high=dfpl['High'],
                             low=dfpl['Low'],
                             close=dfpl['Close']),
              row=1, col=1)

#Add markers for trade entry points on the same subplot
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['support_point'], mode="markers",
                         marker=dict(size=8, color="green"),
                         name="support point"),
              row=1, col=1)

#Add markers for trade entry points on the same subplot
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['resistant_point'], mode="markers",
                         marker=dict(size=8, color="red"),
                         name="resistant point"),
              row=1, col=1)

fig.update_layout(width=1200, height=800, sliders=[])
fig.show()

In [48]:
class KeyLevelStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    setup_duration_gap = 60
    
    def init(self):
        self.current_day = None
        self.setup_duration = None
        self.order_placed_time = None

        super().init()

    def reset_order(self, current_bar_date):
        self.current_day = current_bar_date

        # delete placed order at the end of the day if not executed
        for order in self.orders:
            order_executed= False
            for trade in self.trades:
                if trade.tag == order.tag:
                    order_executed = True

            if order_executed == False:
                order.cancel()

        self.order_placed_time = None
        
    def get_tag(self):
        return f"{self.order_placed_time}"

    def get_sl(self):
        return self.data['ATR'][-1]*1.5
    
    def next(self):
        super().next()

        current_bar_date = self.data.index[-1].date()
        if self.current_day != current_bar_date:
            self.reset_order(current_bar_date)

        if len(self.orders)==0:
            if self.data['TotalSignal'][-1]==1:
                planned_entry_price = self.data['valid_resistant_levels'][-1]
                sl = self.get_sl()
                tp = self.TPcoef * sl

                sl1 = planned_entry_price + sl
                tp1 = planned_entry_price - tp
                position_size = int((self.risk_percent * self.equity) / sl)

                self.order_placed_time = self.data.index[-1]
                self.sell(sl=sl1, tp=tp1, size=position_size, limit=planned_entry_price, tag=self.get_tag())
                
            if self.data['TotalSignal'][-1]==2:
                planned_entry_price = self.data['valid_support_levels'][-1]
                sl = self.get_sl()
                tp = self.TPcoef * sl

                sl1 = planned_entry_price - sl
                tp1 = planned_entry_price + tp
                position_size = int((self.risk_percent * self.equity) / sl)

                self.order_placed_time = self.data.index[-1]
                self.buy(sl=sl1, tp=tp1, size=position_size, limit=planned_entry_price, tag=self.get_tag())

In [72]:
start_date_chart = '2021-1-1 00:00'
end_date_chart = '2026-12-30 00:00'
global_dfopt = df_h4.loc[start_date_chart : end_date_chart]

bt = Backtest(global_dfopt, KeyLevelStrat, cash=10000, margin=0.005, finalize_trades=True) #0.0002
stats = bt.run()

print(stats)
bt.plot()


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2021-07-06 12:00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2023-01-18 16:00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.c

Start                     2021-01-04 00:00:00
End                       2026-09-16 00:00:00
Duration                   2081 days 00:00:00
Exposure Time [%]                    10.30221
Equity Final [$]                   8756.87679
Equity Peak [$]                    10226.6322
Return [%]                          -12.43123
Buy & Hold Return [%]               -15.49709
Return (Ann.) [%]                    -2.22451
Volatility (Ann.) [%]                   4.304
CAGR [%]                             -2.30298
Sharpe Ratio                         -0.51685
Sortino Ratio                         -0.6963
Calmar Ratio                         -0.15478
Alpha [%]                           -12.57337
Beta                                 -0.00917
Max. Drawdown [%]                   -14.37184
Avg. Drawdown [%]                    -4.46238
Max. Drawdown Duration     1405 days 16:00:00
Avg. Drawdown Duration      407 days 20:00:00
# Trades                                   67
Win Rate [%]                      

GridPlot(id='p6638', ...)

In [55]:
# using m15 market structure for entry
def apply_total_signal(df):
    # Initialize the 'TotalSignal' column
    df_signal_list = []
    valid_support_levels_list = []
    valid_resistant_levels_list = []

    for i in range(0, len(df)):
        valid_breakout_resistant_levels = []
        valid_breakout_support_levels = []
        signal = 0
        # valid resistants levels are when valid breakout and levels touched
        valid_resistant_level = None
        valid_support_levels = None

        # short
        c_short_daily_direction = df['market_state_d1'].iloc[i-1] == 'BEARISH'
        if c_short_daily_direction:
            # breakout_candles_below candle needs to be below resistant level for the breakout to be valid
            valid_breakout_resistant_levels = df['valid_breakout_resistant_levels'].iloc[i-1]
            if len(valid_breakout_resistant_levels) > 0:
                valid_resistant_level = min(valid_breakout_resistant_levels)
                signal = 1

        # long
        c_long_daily_direction = df['market_state_d1'].iloc[i-1] == 'BULLISH'
        if c_long_daily_direction:
            # breakout_candles_above candle needs to be above support level for the breakout to be valid
            valid_breakout_support_levels = df['valid_breakout_support_levels'].iloc[i-1]
            if len(valid_breakout_support_levels) > 0:
                valid_support_levels = max(valid_breakout_support_levels)
                signal = 2

        df_signal_list.append(signal)
        valid_resistant_levels_list.append(valid_resistant_level)
        valid_support_levels_list.append(valid_support_levels)

    return df_signal_list, valid_resistant_levels_list, valid_support_levels_list

df_h4['TotalSignal'], df_h4['valid_resistant_levels'], df_h4['valid_support_levels'] = apply_total_signal(df=df_h4)

In [56]:
len(df_h4[df_h4.TotalSignal != 0])

2996

In [57]:
start_date_chart = '2021-1-1 00:00'
end_date_chart = '2026-12-30 00:00'
global_dfopt = df_h4.loc[start_date_chart : end_date_chart]

bt = Backtest(global_dfopt, KeyLevelStrat, cash=10000, margin=0.005, finalize_trades=True) #0.0002
stats = bt.run()

print(stats)
bt.plot()


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2023-01-18 16:00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2024-06-20 08:00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.c

Start                     2021-01-04 00:00:00
End                       2026-09-16 00:00:00
Duration                   2081 days 00:00:00
Exposure Time [%]                     9.69554
Equity Final [$]                   9296.47077
Equity Peak [$]                   10435.65222
Return [%]                           -7.03529
Buy & Hold Return [%]               -15.49709
Return (Ann.) [%]                    -1.22867
Volatility (Ann.) [%]                 4.12361
CAGR [%]                             -1.27224
Sharpe Ratio                         -0.29796
Sortino Ratio                        -0.41359
Calmar Ratio                         -0.11255
Alpha [%]                            -7.28787
Beta                                  -0.0163
Max. Drawdown [%]                   -10.91625
Avg. Drawdown [%]                    -2.89257
Max. Drawdown Duration     1405 days 16:00:00
Avg. Drawdown Duration      339 days 12:00:00
# Trades                                   58
Win Rate [%]                      

GridPlot(id='p3525', ...)

In [70]:
# add some sl tp management

class RandomStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    trades_list = None
    stoploss_window = 150

    def is_signal(self, date):
        for i in range(0, len(self.trades_list)):
            if self.trades_list[i] == date:
                return True
        return False

    def get_highest_high(self):
        return self.data.High[len(self.data.High) - self.stoploss_window : len(self.data.High)].max()
    
    def get_lowest_low(self):
        return self.data.Low[len(self.data.Low) - self.stoploss_window : len(self.data.Low)].min()
    
    def init(self):
        super().init()

    def get_sl(self):
            return self.data['ATR'][-1]*1.5

    def next(self):
        super().next()
        if self.is_signal(self.data.index[-1]) and len(self.trades) == 0 and len(self.closed_trades) < 67:
            direction = randint(1,2)
            if direction == 1:
                planned_entry_price = self.data.Close[-1]
                sl = self.get_sl()
                tp = self.TPcoef * sl

                sl1 = planned_entry_price - sl
                tp1 = planned_entry_price + tp
                position_size = int((self.risk_percent * self.equity) / sl)
                # max_affordable_size = int((self.equity * 0.99) / planned_entry_price)
                # print(f'{position_size} {max_affordable_size}')

                self.buy(sl=sl1, tp=tp1, size=position_size)
            else:
                planned_entry_price = self.data.Close[-1]
                sl = self.get_sl()
                tp = self.TPcoef * sl

                sl1 = planned_entry_price + sl
                tp1 = planned_entry_price - tp
                position_size = int((self.risk_percent * self.equity) / sl)
                # max_affordable_size = int((self.equity * 0.99) / planned_entry_price)
                # print(f'{position_size} {max_affordable_size}')

                self.sell(sl=sl1, tp=tp1, size=position_size)

In [71]:
trades_list = []
while len(trades_list) != 75:
    random_index = randint(0, len(df_h4))
    candle = df_h4.iloc[random_index]
    # exclu Saturday and Sunday
    if candle.name.weekday() < 5:
            trades_list.append(candle.name)
trades_list.sort()

bt = Backtest(df_h4, RandomStrat, cash=10000, margin=0.005, finalize_trades=True) #0.0002
stats = bt.run(trades_list=trades_list)

print(stats)

bt.plot()

Start                     2021-01-04 00:00:00
End                       2026-09-16 00:00:00
Duration                   2081 days 00:00:00
Exposure Time [%]                     9.25739
Equity Final [$]                  11041.63839
Equity Peak [$]                   11122.79107
Return [%]                           10.41638
Buy & Hold Return [%]               -15.49709
Return (Ann.) [%]                     1.69342
Volatility (Ann.) [%]                 4.99154
CAGR [%]                              1.75438
Sharpe Ratio                          0.33926
Sortino Ratio                         0.54189
Calmar Ratio                          0.24411
Alpha [%]                            10.70034
Beta                                  0.01832
Max. Drawdown [%]                    -6.93712
Avg. Drawdown [%]                    -2.09267
Max. Drawdown Duration      927 days 00:00:00
Avg. Drawdown Duration      112 days 00:00:00
# Trades                                   67
Win Rate [%]                      

GridPlot(id='p6355', ...)

In [73]:
i = 0
expectancy_list = []
winrate_list = []

while len(expectancy_list) < 1000:
    trades_list = []
    while len(trades_list) != 75:
        random_index = randint(0, len(df_h4)-1)
        candle = df_h4.iloc[random_index -1]
        # exclu Saturday and Sunday
        if candle.name.weekday() < 5:
                trades_list.append(candle.name)
    trades_list.sort()

    bt = Backtest(df_h4, RandomStrat, cash=10000, margin=0.009, finalize_trades=True) #0.0002
    stats = bt.run(trades_list=trades_list)
    if stats['# Trades'] == 67:
        expectancy_list.append(stats['Expectancy [%]'])
        winrate_list.append(stats['Win Rate [%]'])
        print(f'{i} {len(expectancy_list)} {stats['Expectancy [%]']} Done')
    i += 1

print(expectancy_list)

1 1 -0.0715440352309447 Done
2 2 -0.05128320048522492 Done
4 3 0.025641877143048705 Done
5 4 0.050813136793614544 Done
6 5 -0.008306598690873045 Done
8 6 -0.034402326745832094 Done
10 7 -0.04279491211578545 Done
11 8 0.057260441106811215 Done
12 9 -0.07089470949934845 Done
13 10 -0.0017555385589237927 Done
15 11 -0.02596168529154396 Done
16 12 0.06516743860502806 Done
17 13 -0.04830341507762861 Done
18 14 -0.14442486910771654 Done
20 15 -0.06970817100576394 Done
21 16 -0.03213047694923122 Done
22 17 -0.0829758031000159 Done
23 18 -0.07337450244142794 Done
24 19 -0.061590229697112996 Done
25 20 0.007852315016184348 Done
26 21 -0.06307855292120268 Done
27 22 -0.05990387108970166 Done
28 23 0.07454031755648995 Done
29 24 0.03926795517189453 Done
30 25 -0.0232083487111516 Done
31 26 0.021525651130175 Done
32 27 0.11927954230415945 Done
33 28 0.03820600389229229 Done
34 29 -0.06708971366799694 Done
35 30 -0.07923583769730676 Done
36 31 0.00837129144104382 Done
37 32 0.10802001979833699 Done

In [74]:
print(len(expectancy_list))

p95 = np.percentile(expectancy_list, 95)                  # ngưỡng phân vị 95
print(f"p95 = {p95}")
# max_expectancy = max(expectancy_list_2)
print(f"max = {max(expectancy_list)}")
mean_expectancy = np.mean(expectancy_list)
print(f"mean = {mean_expectancy}")
print(f"min = {min(expectancy_list)}")
print(f"gap = {-0.0661 - mean_expectancy}")

df_result = pd.DataFrame({'expectancy' : expectancy_list})
rank = (df_result['expectancy'] <= -0.0661).mean() * 100 
print(f"rank = {rank}")
p_value = ((df_result['expectancy'] >= -0.0661).sum() + 1) / (len(df_result['expectancy']) + 1)
print(f"p_value = {p_value}")

print(len(winrate_list))

p95_winrate = np.percentile(winrate_list, 95)                  # ngưỡng phân vị 95
print(f"p95 = {p95_winrate}")
# max_expectancy = max(expectancy_list_2)
print(f"max = {max(winrate_list)}")
mean_winrate = np.mean(winrate_list)
print(f"mean = {mean_winrate}")
print(f"min = {min(winrate_list)}")
print(f"gap = {45.09804 - mean_winrate}")



1000
p95 = 0.10501339909562271
max = 0.20787433085652868
mean = -0.0190982289240166
min = -0.2206939634664572
gap = -0.047001771075983405
rank = 27.800000000000004
p_value = 0.7222777222777222
1000
p95 = 40.298507462686565
max = 47.76119402985074
mean = 31.929850746268652
min = 16.417910447761194
gap = 13.168189253731345
